# __ML in der Logistik__
# Exercise 05 - SVM

__Goal__: Understand the concept of SVM, the different kernel functions, the effet of parameters, and practice in Python.


__Data and application (dataset 1)__: The dataset contains the SOG and COG information of 674 AIS messages from 20 trips recorded on 2 routes in North Europa (Rotterdam > Hamburg and Kiel > Gdynia). We want to build a SVM model to predict which route the ship was on from the SOG and COG information.

__Data and application (dataset 2)__: The dataset contains the location (latitude and longitude) of 674 AIS messages from 20 trips recorded on 2 routes in North Europa (Rotterdam > Hamburg and Kiel > Gdynia). We want to build a SVM model to predict which route the ship was on from the latitude and longitude information.



### Contents

* [1 Theory: basics on SVM (10 min)](#t1)
* [2 Practice: linear and Gaussian kernels and decision boundary](#t2)
    * [2.1 Baseline and other models (10 min)](#t21)
    * [2.2 Decision boundary visualizer (5 min)](#t22)
    * [2.3 SVC algorithm: linear kernel (15 min)](#t23)
    * [2.4 SVC algorithm: Gaussian kernel (15 min)](#t24)

## 1 Theory: basics on SVM (10 min) <a class="anchor" id="t1"></a>

_Explain simply the concept of the SVM model for a binary classification and the kernel trick._

_What is the effect of the parameter_ ``c`` _(complexity)?_

_Write down the kernel functions for the linear kernel, the polynomial kernel and the Gaussian kernel._

_For a linear kernel with two attributes, what is the effect of the parameter c on the largest margin separating line?_

_For a Gaussian kernel, what is the effect of the parameter_ ``gamma`` _on the model?_

## 2 Practice: linear and Gaussian kernels and decision boundary <a class="anchor" id="t2"></a>

### 2.1 Baseline and other models (10 min) <a class="anchor" id="t21"></a>

For this task we will use the dataset ``05-SVM_1_TRAIN.csv``. It contains some AIS data, groupe by trips, for the route Rotterdam > Hamburg (8 trips) and Kiel > Gdynia (8 trips). The two attributes are the SOG (speed over ground) and COG (course over ground, the direction along which the ship is to be steered, expressed in degree: [0, 360]). The class is the Route, a binary attribute which can take the values ``ROT-HAM`` and ``KIE-GDY``.

Note: for this task, the train and test datasets are already splitted and given as ``05-SVM_1_TRAIN.csv`` and ``05-SVM_1_TEST.csv``.

In [ ]:
# Load train and test datasets

import pandas as pd
df_train = pd.read_csv('05-SVM_1_TRAIN.csv', na_values = '?')
print(df_train.head())
print(df_train.dtypes)

df_test = pd.read_csv('05-SVM_1_TEST.csv', na_values = '?')

__From the first overview of the dataset, transform the attributes that need transforming into a numerical or categorical type (do it for both train and test sets). (1 point/dataset)__

In [ ]:
# Transform the types of the attributes



Here is a picture of the two routes on a world map (left: Rotterdam > Hamburg, right: Kiel > Gdynia):

![text](05-coordinates_map.PNG)

__Visualize the 2 attributes of the dataset, using the visualize function that we already saw in previous tasks.__

In [ ]:
# Visualize function (just execute this code)
import numpy as np

def plot_2att(df, att1, att2, clas_att):
    import matplotlib.pyplot as plt
    '''
    Function for plotting two attributes colored by a class attribute
    
    df: dataframe, the dataset
    att1, att2: strings, the names of the two attributes to be plotted
    clas_att: string, the name of the attribute used for coloration !! must be categorical or object
    '''
    
    df2 = df.copy()
    
    if df2[clas_att].dtype.name in ['object', 'int64', 'float64']:
        df2[clas_att] = df2[clas_att].astype('category')
    elif df2[clas_att].dtype.name != 'category':
        raise Exception('The class attribute must be categorical or object.')
    
    # list_clas: list of the values of the class attribute (TripID)
    list_clas = df2[clas_att].cat.categories.tolist()

    # Create one list for each clas_att value in the array tuples
    tuples = []
    for i in list_clas:
        tuples.append(
            [df2.loc[df[clas_att] == i][att1].values, df2.loc[df[clas_att] == i][att2].values]
        )

    plt.figure(figsize=(12,8))
    j = 0
    for i in tuples:
        label = str(clas_att) + ' = ' + str(list_clas[j])
        j = j + 1
        plt.plot(np.asarray(i[0]), np.asarray(i[1]), 'x', label = label) # use np.asarray in case one of the attribute is
                                                          # categorical, i[0] would return a category and not an array
    plt.legend(loc = 'upper left')
    title = att1 + ' vs. ' + att2 + ' colored by ' + clas_att
    plt.title(title)

In [ ]:
# Visualize the dataset



We don't know if there is a solution for this problem, but we can try out couple of methods to find if it is possible to predict the Route of one AIS message, only from the SOG and the COG. Among other methods, we can use a baseline classifier to determine if a solution exists for the problem.

A baseline classifier is a classifier that makes predictions using very simple rules. Comparing the results of a fancy ML model with a basic baseline one is useful to determine if the ML model is really good, or if the result is due to luck.

In Python, the algorithm ``DummyClassifier`` from the library ``sklearn`` allows to build simple baseline models.
This algorithm can take different values for the parameter ``strategy``, which characterises which rule is used to classify the dataset:
+ ``stratified``: generates predictions by respecting the training set's class distribution.
+ ``most_frequent``: always predicts the most frequent label in the training set.
+ ``prior``: always predicts the class that maximizes the class prior (like most_frequent) and predict_proba returns the class prior.
+ ``uniform``: generates predictions uniformly at random.
+ ``constant``: always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class.

__For this task, try the baseline classifier with the values ``strategy = 'most_frequent'`` and ``strategy = 'stratified'``. Save the performance for both classifiers.__
Note: always use ``random_state = 1``.

Note: the algorithm ``DummyClassifier`` contains a bug which makes it impossible to use with Pandas dataframes. A small change is to make on the dataset for the use of this classifier (``check_X_y`` is used here, a function from the ``sklearn`` library that transforms the data into a processable form), and for this reason, part of the code of the next cell is already given. Make the necessary changes to obtain the performance you need.

In [ ]:
# Apply baseline classifiers

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error

from sklearn.utils import check_X_y

# ???

X_converted, y_converted = check_X_y(X = df_train[x], y = df_train[y])
dummy.fit(X = X_converted, y = y_converted)

# ???

__According to the type of the task, which performance measurement should we compute? The__ ``accuracy_score`` __or the__ ``mean_absolute_error`` __? Set the value of the chosen performance to 1 in the following cell. (1 point).__

In [ ]:
# Performance

t21_accuracy_score = 
t21_mean_absolute_error = 

__Try out some other known classifiers with different parameters (Decision tree, random forest, neural networks, ...).__

In [ ]:
# Try more classifiers



_If we take one AIS message (one red point on the map), do you think it is possible to determine the Route taken from only the SOG and COG values at that point? Why?_

_Is the problem linearly separable? If yes, where is the separation?_

_With the classic classifiers, what is the best performance you can reach? With which classifier and which parameters? Is this performance due to luck or is the classifier efficient?_

__Is the problem linearly separable? Set the value to 0 if the problem is not linearly separable, 1 otherwise. (2 points)__

In [ ]:
# Is the problem linearly separable?

t21_is_linearly_separable = 

### 2.2 Decision boundary visualizer (5 min) <a class="anchor" id="t22"></a>

In this exercise, we will study the difference the values of the parameters of the SVM model make, by looking a graphs plotting the decision boundary of the model.

The function to plot the decision boundary is quite complicated, and it is not asked for the student to understand it. Just execute the code and use the function as shown in the example.

The function is ``visualize_boundaries`` and its parameters are the training dataset, the test dataset, and the values for ``x`` and ``y`` as they were used in the very first exercise (``01-Basics-preprocessing``).
To use this function with your model, just modify the line of code for the model and replace it with the model you want to use, with the correct parameters. We initialized the function using the ``KNeighborsClassifier`` algorithm from the ``sklearn`` library.

Note: the decision boundary visualizer might take a few seconds or minutes to execute. It is normal, and for this reason we will work from already plotted graphs in future parts of the task. It is just important that you understand how the boundary visualizer is used and what is its output.

In [ ]:
# Decision boundary function: just execute the code

import numpy as np
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier

def visualize_boundaries(df, x, y):
    
    '''
    df: dataframe, the dataset to plot
    x: array containing the names of the predictive variables (size = 2)
    y: array containing the name of the predicted variable
    '''
    
    # model: change this with another algorithm to visualize
    # -------------------------------------------------------------------
    model = KNeighborsClassifier(n_neighbors = 25)
    # -------------------------------------------------------------------
    
    df2 = df.copy()
    
    if y == ['Route']:
        df2['class'] = df2['Route'].apply(lambda x: 1 if x == 'ROT-HAM' else 0)
    
    model.fit(df2[x], df2['class'].values.ravel())
    
    # title for the plot
    title = model
    
    list_clas = df2['class'].cat.categories.tolist()
    # Create one list for each clas_att value in the array tuples
    tuples = []
    for i in list_clas:
        tuples.append(
            [df2.loc[df2['class'] == i][x[0]].values, df2.loc[df2['class'] == i][x[1]].values]
    )

    # Set-up 2x2 grid for plotting.
    fig, ax = plt.subplots()

    x_min, x_max = df2[x][x[0]].min() - 1, df2[x][x[0]].max() + 1
    y_min, y_max = df2[x][x[1]].min() - 1, df2[x][x[1]].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.8)

    j = 0
    for i in tuples:
        j = j + 1
        plt.plot(np.asarray(i[0]), np.asarray(i[1]), 'x') # use np.asarray in case one of the attribute is
                                                          # categorical, i[0] would return a category and not an array

    ax.set_xlim(xx.min(), xx.max())
    ax.set_ylim(yy.min(), yy.max())

    ax.set_xlabel(x[0])
    ax.set_ylabel(x[1])

    ax.set_xticks(())
    ax.set_yticks(())
    ax.set_title(title)

    plt.show()

In [ ]:
# How to use the visualize_boundaries function

x = ['SOG', 'COG']
y = ['Route']

visualize_boundaries(df_train, x, y)

__Change the model of the boundary classifier with the one you found gave the best performance in section 2.1.__ Do not forget to re-execute the code of the function after you modified it.

_Explain the graph you see and how a new instance would be classified._

### 2.3 SVC algorithm: linear kernel (15 min) <a class="anchor" id="t23"></a>

To build a SVM model, we will use the algorithm ``SVC`` from the ``sklearn`` library.

Among others, one of the parameters of this algorithm is ``kernel``, it used to select the kernel function to use for classification. In this first task, we will set this parameter to ``kernel = 'linear'``, to use a linear kernel.

We keep the dataset from the part 2.1.

__Write the code to build this model and save the performance. Use the different training and test sets given in part 2.1. Do not forget the parameters__ ``random_state = 1``.

In [ ]:
# Apply the SVC algorithm with linear kernel

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error



The parameter ``C`` is the complexity. The default value for this parameter is 1.0. __Try out other values (in the range [0.000001; 1000]) for this parameter and save the performance for each.__

In [ ]:
# Try other values for C



What is the value for ``C`` which gives the best accuracy? __Write it down in the following cell (if several values give the maximum accuracy, just write one of them). (1 point)__

In [ ]:
# Best value for C

t23_C = 

We will try this same algorithm with the same parameters with another dataset. This dataset contains the same AIS messages as the previous one, but instead of using SOG and COG for classification, it uses the latitude and longitude values. As a reminder, we want to predict the route, and the latitude and longitude values might probably give good information on this attribute...

__Build the same model with the new dataset, and save the performance. Try out different values for__ ``C``.

In [ ]:
# Import the latitude and longitude dataset

df_train = pd.read_csv('05-SVM_2_TRAIN.csv', na_values = '?')
print(df_train.head())
print(df_train.dtypes)

df_test = pd.read_csv('05-SVM_2_TEST.csv', na_values = '?')

df_train['Route'] = df_train['Route'].astype('category')
df_test['Route'] = df_test['Route'].astype('category')

In [ ]:
# Build the model



What is the value for ``C`` which gives the best accuracy? __Write it down in the following cell (if several values give the maximum accuracy, just write one of them). (1 point)__

In [ ]:
# Best value for C

t23_C2 = 

Finally, here are the graphs of the decision boundary for the two datasets, using the SVC algorithm with a linear kernel, with different values for the parameter ``C``:

![text](05-boundary-linear.png)

_Compare the performance of the classification with the default parameters to the baseline, and make a conclusion about your answer at the question "Is the problem separable?"._

_What is the effect of C on the boundary of the classification for the dataset using SOG-COG? Can you identify the effect you mentioned in the theoretical part, for a lower C and a higher C? If not, why?_

_Is the effect of C clearer with the dataset using lat-long? Why? What is the effect of C?_

### 2.4 SVC algorithm: Gaussian kernel (15 min) <a class="anchor" id="t24"></a>

We keep working with the datasets using SOG-COG for classification.

Now, we will be using the Gaussian kernel for the classifier. For the ``SVC`` algorithm, this parameter is ``kernel = 'rbf'``.

__Write the code to build this model with default parameters and save the performance. Use the different training and test sets given in part 2.1. Do not forget the parameters__ ``random_state = 1``.

In [ ]:
# Reimport the datasets into df

df_train = pd.read_csv('05-SVM_1_TRAIN.csv', na_values = '?')
df_test = pd.read_csv('05-SVM_1_TEST.csv', na_values = '?')

In [ ]:
# Apply the SVC algorithm with gaussian kernel

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error



__Try the same algorithm with the value for the parameter__ ``C = 1.0``.

In [ ]:
# Apply the SVC algorithm with gaussian kernel with C = 1.0

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error



The parameter for gamma in this algorithm is called ``gamma``. We will study the effect of this parameter on the model.  The default value of this parameter is 1 / n_features, in our case, 1/2 = 0.5.

__Keeping__ ``C = 1.0``__, try out different values for gamma. You can try in the range [0.001 ; 100].__

In [ ]:
# Try different values for gamma

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error



What is the value for gamma which gives the best accuracy? __Write it down in the following cell (if several values give the maximum accuracy, just write one of them). (1 point)__

In [ ]:
# Best value for gamma

t24_gamma = 

Here are the graphs of the decision boundary for different models with ``C = 1`` and different values for ``gamma``:

![text](05-boundary-rbf-c1.png)

_Do the visualizations of the models match with the performances you measured before?_

_From the visualization of the boundaries, which model seems to be the best for this dataset? Why? Does it make sense from the performances?_

_Explain simply the effect of gamma on the decision boundary of the model._

__Choose the best value for__ ``gamma`` __, and use it to try out several values for__ ``C`` __. The goal is to find the best set of parameters.__

In [ ]:
# Find the best set of parameters

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error



Here are the graphs of the boundary visualizers for different models with ``gamma = 0.001`` and ``gamma = 0.01``:

![text](05-boundary-rbf-g0001.png)

![text](05-boundary-rbf-g001.png)

_Compare the models and argue for your choice of parameters for the best model, using the decision boundary visualizer and the measured performances._

_Compare the models with the baseline. Would you say they are efficient classifiers?_